# CocinaAI — EDA e Ingeniería de Features

**Proyecto:** CocinaAI — Tu chef de alacena

**Fuentes:** Spoonacular API (recetas mexicanas) + TheMealDB (área Mexican)

**Objetivo:** Analizar patrones de ingredientes en cocina mexicana para respaldar el diseño del sistema de recomendación.

### Imports

In [1]:
!pip install requests pandas numpy plotly scikit-learn -q

In [2]:
import json
import time
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import scipy.sparse
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [3]:
from google.colab import userdata
spoonacular_api = userdata.get('SPOONACULAR_API')
print('API key cargada:', 'OK' if spoonacular_api else 'ERROR - revisa Secrets')

API key cargada: OK


### Variables globales

In [4]:
N_SPOONACULAR = 100  # recetas de Spoonacular
# TheMealDB: solo area Mexican (autenticas), sin categorias genericas
THEMEALDB_AREAS = ['Mexican']

## 1. Funciones de ingesta

In [5]:
def get_spoonacular_recipes(cuisine, number, api_key):
    """Obtiene recetas mexicanas desde Spoonacular."""
    url = 'https://api.spoonacular.com/recipes/complexSearch'
    params = {
        'cuisine': cuisine, 'number': number,
        'addRecipeInformation': True, 'addRecipeNutrition': False,
        'apiKey': api_key
    }
    resp = requests.get(url, params=params, timeout=15)
    return resp.json().get('results', [])


In [6]:
def parse_spoonacular(recipe):
    """Normaliza receta de Spoonacular."""
    ingredients = recipe.get('extendedIngredients', [])
    names = [i.get('name','').lower().strip() for i in ingredients if i.get('name')]
    return {
        'id': f"sp_{recipe.get('id')}",
        'titulo': recipe.get('title',''),
        'fuente': 'Spoonacular',
        'tiempo_minutos': recipe.get('readyInMinutes'),
        'porciones': recipe.get('servings'),
        'num_ingredientes': len(names),
        'ingredientes': names,
        'vegana': recipe.get('vegan', False),
        'vegetariana': recipe.get('vegetarian', False),
        'sin_gluten': recipe.get('glutenFree', False),
        'url': recipe.get('sourceUrl',''),
        'imagen': recipe.get('image',''),
    }


In [7]:
def get_themealdb_by_area(area):
    """Obtiene recetas de TheMealDB por area geografica."""
    url = f'https://www.themealdb.com/api/json/v1/1/filter.php?a={area}'
    resp = requests.get(url, timeout=15)
    return resp.json().get('meals', []) or []

def get_themealdb_detail(meal_id):
    """Obtiene detalle completo de una receta de TheMealDB."""
    url = f'https://www.themealdb.com/api/json/v1/1/lookup.php?i={meal_id}'
    resp = requests.get(url, timeout=15)
    meals = resp.json().get('meals', [])
    return meals[0] if meals else {}


In [8]:
def parse_themealdb(meal):
    """Normaliza receta de TheMealDB."""
    ingredients = []
    for i in range(1, 21):
        ing = meal.get(f'strIngredient{i}', '')
        if ing and ing.strip():
            ingredients.append(ing.lower().strip())
    category = meal.get('strCategory','').lower()
    tags = (meal.get('strTags') or '').lower()
    is_veg = any(v in category or v in tags for v in ['vegetarian','vegan','veggie'])
    return {
        'id': f"mdb_{meal.get('idMeal')}",
        'titulo': meal.get('strMeal',''),
        'fuente': 'TheMealDB',
        'tiempo_minutos': None,
        'porciones': None,
        'num_ingredientes': len(ingredients),
        'ingredientes': ingredients,
        'vegana': 'vegan' in category,
        'vegetariana': is_veg,
        'sin_gluten': False,
        'url': meal.get('strSource',''),
        'imagen': meal.get('strMealThumb',''),
    }


## 2. Descarga de datos

In [9]:
# Spoonacular
print('Descargando Spoonacular...')
raw_sp = get_spoonacular_recipes('mexican', N_SPOONACULAR, spoonacular_api)
sp_list = [parse_spoonacular(r) for r in raw_sp]
print(f'  Spoonacular: {len(sp_list)} recetas')

Descargando Spoonacular...
  Spoonacular: 100 recetas


In [10]:
# TheMealDB — solo area Mexican
print('Descargando TheMealDB (area Mexican)...')
mdb_ids = set()
mdb_list = []

for area in THEMEALDB_AREAS:
    for m in get_themealdb_by_area(area):
        if m['idMeal'] not in mdb_ids:
            mdb_ids.add(m['idMeal'])
            detail = get_themealdb_detail(m['idMeal'])
            if detail:
                mdb_list.append(parse_themealdb(detail))
            time.sleep(0.1)

print(f'  TheMealDB: {len(mdb_list)} recetas')

Descargando TheMealDB (area Mexican)...
  TheMealDB: 6 recetas


In [11]:
# Fusion — Spoonacular primero para ganar en deduplicacion
df_sp  = pd.DataFrame(sp_list)  if sp_list  else pd.DataFrame()
df_mdb = pd.DataFrame(mdb_list) if mdb_list else pd.DataFrame()

recipes_df = pd.concat([df_sp, df_mdb], ignore_index=True)
print(f'Antes de limpiar: {len(recipes_df)} recetas')
print(recipes_df['fuente'].value_counts())

Antes de limpiar: 106 recetas
fuente
Spoonacular    100
TheMealDB        6
Name: count, dtype: int64


In [12]:
print(f'df_sp shape: {df_sp.shape}')
print(f'df_mdb shape: {df_mdb.shape}')
print()
print('Primeras filas de df_sp:')
print(df_sp[['titulo','fuente']].head(3))

df_sp shape: (100, 12)
df_mdb shape: (6, 12)

Primeras filas de df_sp:
                                      titulo       fuente
0                         Corn Avocado Salsa  Spoonacular
1  Cheesy Chicken Enchilada Quinoa Casserole  Spoonacular
2                         Homemade Guacamole  Spoonacular


## 3. Limpieza

In [13]:
n = len(recipes_df)

# 1. Sin ingredientes
recipes_df = recipes_df[recipes_df['num_ingredientes'] > 0]
print(f'Eliminadas sin ingredientes: {n - len(recipes_df)}')

# 2. Duplicados por titulo (Spoonacular gana al ir primero)
n = len(recipes_df)
recipes_df = recipes_df.drop_duplicates(subset='titulo', keep='first')
print(f'Eliminadas por titulo duplicado: {n - len(recipes_df)}')

# 3. Outliers de tiempo — solo eliminar si tiempo es muy extremo
n = len(recipes_df)
recipes_df = recipes_df[
    recipes_df['tiempo_minutos'].isna() |
    (recipes_df['tiempo_minutos'] <= 480)
]
print(f'Eliminadas por tiempo > 480 min: {n - len(recipes_df)}')

# 4. Imputar tiempo faltante (TheMealDB no tiene tiempo)
tiempo_mediana = recipes_df['tiempo_minutos'].median()
recipes_df['tiempo_minutos'] = recipes_df['tiempo_minutos'].fillna(tiempo_mediana)
recipes_df['porciones'] = recipes_df['porciones'].fillna(4)

recipes_df = recipes_df.reset_index(drop=True)
print(f'\nRecetas limpias: {len(recipes_df)}')
print(recipes_df['fuente'].value_counts())

Eliminadas sin ingredientes: 100
Eliminadas por titulo duplicado: 0
Eliminadas por tiempo > 480 min: 0

Recetas limpias: 6
fuente
TheMealDB    6
Name: count, dtype: int64


## 4. Ingeniería de variables

### 4.1 Dificultad

In [14]:
def clasificar_dificultad(num_ing, tiempo):
    score = 0
    if num_ing > 12: score += 2
    elif num_ing > 7: score += 1
    if tiempo > 60: score += 2
    elif tiempo > 30: score += 1
    return 'Facil' if score <= 1 else ('Media' if score <= 3 else 'Dificil')

recipes_df['dificultad'] = recipes_df.apply(
    lambda r: clasificar_dificultad(r['num_ingredientes'], r['tiempo_minutos']), axis=1
)
recipes_df['dificultad'].value_counts()

,count
dificultad,
Media,4
Facil,2


### 4.2 Score de alacena

In [15]:
# Top 60 ingredientes mas frecuentes del dataset
all_ings = [i for sublist in recipes_df['ingredientes'] for i in sublist]
ingredient_counts = Counter(all_ings)
top_60 = set([ing for ing, _ in ingredient_counts.most_common(60)])

print('Top 20 ingredientes mas frecuentes:')
print([ing for ing, _ in ingredient_counts.most_common(20)])

Top 20 ingredientes mas frecuentes:
['garlic', 'olive oil', 'cumin', 'avocado', 'sour cream', 'corn tortillas', 'chicken breasts', 'lime', 'smoked paprika', 'shredded mexican cheese', 'beef', 'onions', 'chorizo', 'allspice', 'cloves', 'cinnamon stick', 'bay leaves', 'oregano', 'ancho chillies', 'balsamic vinegar']


In [16]:
# Proporcion de ingredientes comunes de alacena
recipes_df['prop_alacena'] = recipes_df['ingredientes'].apply(
    lambda ings: round(len(set(ings) & top_60) / max(len(ings), 1), 3)
)

# Pocos ingredientes (mas facil de tener todo)
recipes_df['pocos_ingredientes'] = (recipes_df['num_ingredientes'] <= 7).astype(int)

# Receta rapida
recipes_df['es_rapida'] = (recipes_df['tiempo_minutos'] <= 35).astype(int)

# Score compuesto
recipes_df['score_alacena'] = (
    0.60 * recipes_df['prop_alacena'] +
    0.25 * recipes_df['pocos_ingredientes'] +
    0.15 * recipes_df['es_rapida']
).round(3)

print('Score alacena:')
print(recipes_df['score_alacena'].describe().round(3))

Score alacena:
count   6.00
mean    0.63
std     0.11
min     0.55
25%     0.60
50%     0.60
75%     0.60
max     0.85
Name: score_alacena, dtype: float64


In [17]:
# Top 10 recetas mas aprovechables
recipes_df[['titulo','fuente','num_ingredientes','score_alacena','dificultad']]\
    .sort_values('score_alacena', ascending=False).head(10)

,titulo,fuente,num_ingredientes,score_alacena,dificultad
2,Chicken Enchilada Casserole,TheMealDB,4,0.85,Facil
0,Braised Beef Chilli,TheMealDB,17,0.60,Media
1,Cajun spiced fish tacos,TheMealDB,12,0.60,Facil
4,Crock Pot Chicken Baked Tacos,TheMealDB,14,0.60,Media
5,Stuffed Bell Peppers with Quinoa and Black Beans,TheMealDB,15,0.60,Media
3,Chickpea Fajitas,TheMealDB,13,0.55,Media


### 4.3 Normalizacion

In [18]:
cols_norm = ['num_ingredientes','tiempo_minutos','score_alacena','prop_alacena']
scaler = MinMaxScaler()
norm_vals = scaler.fit_transform(recipes_df[cols_norm].fillna(0))
norm_df = pd.DataFrame(norm_vals, columns=[f'{c}_norm' for c in cols_norm], index=recipes_df.index)
recipes_df = pd.concat([recipes_df, norm_df], axis=1)
print('Normalizacion completa.')

Normalizacion completa.


### 4.4 TF-IDF de ingredientes

In [19]:
# Texto de ingredientes para vectorizacion
recipes_df['ingredientes_texto'] = recipes_df['ingredientes'].apply(
    lambda ings: ' '.join([i.replace(' ','_') for i in ings if i])
)

print(f'Recetas para TF-IDF: {len(recipes_df)}')
print('Muestra:')
print(recipes_df['ingredientes_texto'].head(3).tolist())

Recetas para TF-IDF: 6
Muestra:
['beef onions garlic olive_oil chorizo cumin allspice cloves cinnamon_stick bay_leaves oregano ancho_chillies balsamic_vinegar plum_tomatoes tomato_ketchup dark_brown_sugar borlotti_beans', 'cajun cayenne_pepper white_fish vegetable_oil flour_tortilla avocado little_gem_lettuce spring_onions salsa sour_cream lemon garlic', 'enchilada_sauce shredded_monterey_jack_cheese corn_tortillas chicken_breasts']


In [20]:
# Ajustamos min_df segun el numero de recetas disponibles
n_recetas = len(recipes_df)
min_df = max(1, int(n_recetas * 0.02))  # al menos 2% de recetas
print(f'min_df calculado: {min_df} (para {n_recetas} recetas)')

tfidf = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),
    min_df=min_df
)
tfidf_matrix = tfidf.fit_transform(recipes_df['ingredientes_texto'])
print(f'Matriz TF-IDF: {tfidf_matrix.shape}')

min_df calculado: 1 (para 6 recetas)
Matriz TF-IDF: (6, 128)


In [21]:
# Ingredientes con mayor peso TF-IDF
feature_names = tfidf.get_feature_names_out()
avg_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_tfidf = pd.DataFrame({'ingrediente': feature_names, 'peso': avg_tfidf})\
            .sort_values('peso', ascending=False).head(20)
top_tfidf

,ingrediente,peso
21,chicken_breasts,0.08
35,corn_tortillas,0.08
50,garlic,0.08
4,avocado,0.07
110,sour_cream,0.07
38,cumin,0.07
68,lime,0.07
36,corn_tortillas chicken_breasts,0.07
106,shredded_monterey_jack_cheese corn_tortillas,0.07
105,shredded_monterey_jack_cheese,0.07


## 5. Análisis exploratorio

In [22]:
print('Dataset final:')
print(f'  Total recetas: {len(recipes_df)}')
print(f'  Por fuente:')
print(recipes_df['fuente'].value_counts())
print(f'\nEstadisticas:')
print(recipes_df[['tiempo_minutos','num_ingredientes','score_alacena']].describe().round(2))

Dataset final:
  Total recetas: 6
  Por fuente:
fuente
TheMealDB    6
Name: count, dtype: int64

Estadisticas:
       tiempo_minutos  num_ingredientes  score_alacena
count            0.00              6.00           6.00
mean              NaN             12.50           0.63
std               NaN              4.51           0.11
min               NaN              4.00           0.55
25%               NaN             12.25           0.60
50%               NaN             13.50           0.60
75%               NaN             14.75           0.60
max               NaN             17.00           0.85


In [ ]:
fig = px.pie(recipes_df, names='fuente',
    title='Recetas por fuente',
    color_discrete_map={'Spoonacular':'#B5722A','TheMealDB':'#1D9E75'},
    template='plotly_white')
fig.show()

In [ ]:
# Top ingredientes
top_ing_df = pd.DataFrame(ingredient_counts.most_common(20), columns=['ingrediente','frecuencia'])
fig = go.Figure(go.Bar(x=top_ing_df['ingrediente'], y=top_ing_df['frecuencia'], marker_color='#B5722A'))
fig.update_layout(title='Top 20 ingredientes mas frecuentes', xaxis_tickangle=-45, template='plotly_white')
fig.show()

In [ ]:
# Score alacena por dificultad
sc = recipes_df.groupby('dificultad')['score_alacena'].mean().round(3).reset_index()
fig = go.Figure(go.Bar(x=sc['dificultad'], y=sc['score_alacena'],
    marker_color=['#1D9E75','#EF9F27','#D85A30']))
fig.update_layout(title='Score de alacena promedio por dificultad', template='plotly_white')
fig.show()

In [ ]:
# Scatter ingredientes vs score
fig = px.scatter(recipes_df, x='num_ingredientes', y='score_alacena',
    color='dificultad', hover_name='titulo',
    title='Numero de ingredientes vs Score de alacena',
    color_discrete_map={'Facil':'#1D9E75','Media':'#EF9F27','Dificil':'#D85A30'},
    template='plotly_white')
fig.show()

In [ ]:
print('Hallazgo clave:')
for dif in ['Facil','Media','Dificil']:
    score = recipes_df[recipes_df['dificultad']==dif]['score_alacena'].mean()
    n = len(recipes_df[recipes_df['dificultad']==dif])
    print(f'  {dif}: score promedio = {score:.3f} ({n} recetas)')
print()
print('Conclusion: las recetas de dificultad Facil tienen mayor score de alacena.')
print('Esto valida que el sistema debe priorizar recetas simples para mayor aprovechamiento.')

## 6. Exportación

In [ ]:
export_df = recipes_df.copy()
export_df['ingredientes'] = export_df['ingredientes'].apply(lambda x: ', '.join(x))
export_df = export_df.drop(columns=['ingredientes_texto'], errors='ignore')
export_df.to_csv('recipes_df.csv', index=False)
print(f'recipes_df.csv: {export_df.shape[0]} recetas, {export_df.shape[1]} columnas')

In [ ]:
scipy.sparse.save_npz('tfidf_matrix.npz', tfidf_matrix)
with open('tfidf_feature_names.json', 'w') as f:
    json.dump(list(tfidf.get_feature_names_out()), f)
print('tfidf_matrix.npz exportado')
print('tfidf_feature_names.json exportado')
print()
print('Archivos listos para:')
print('  1. Notebook de Clustering')
print('  2. Tools de la app de Streamlit')

In [ ]:
# Guardar tambien el top_60 para usarlo en las tools
with open('top_ingredientes_alacena.json', 'w') as f:
    json.dump(list(top_60), f, ensure_ascii=False)
print('top_ingredientes_alacena.json exportado')
print(f'  {len(top_60)} ingredientes de alacena tipica mexicana')